<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/04_traduccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Traducción a español (NLLB-200)

Este notebook traduce `dataset_limpio_ingles_balanceado.csv` de inglés a
español, usando NLLB-200 (Meta AI), para la Ronda 2 del experimento:
entrenar y evaluar en español.

Se eligió NLLB-200 sobre Google Translate (riesgo de bloqueo, sin
respaldo académico) y Helsinki-NLP (calidad más literal), porque supera
el estado del arte previo en ~44% y es completamente abierto y
reproducible (Meta AI, NLLB Team, 2022).

Como el dataset ya está filtrado a inglés (notebook 03), se traduce todo
como `eng_Latn → spa_Latn`, sin necesidad de detectar idioma por fila.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!pip install transformers sentencepiece -q

In [15]:
import pandas as pd

RUTA_DATASET = '/content/drive/MyDrive/dataset_limpio_ingles_balanceado.csv'
df = pd.read_csv(RUTA_DATASET)

print('Filas:', len(df))
print('Columnas:', df.columns.tolist())
print(df['label'].value_counts())

Filas: 9534
Columnas: ['text', 'label']
label
1    4767
0    4767
Name: count, dtype: int64


## 1. Cargar el modelo NLLB-200

Se usa la versión "distilled-600M" (más liviana que el modelo completo),
suficiente calidad para este par de idiomas de alto recurso (inglés-español).
Requiere GPU para que el tiempo de procesamiento sea razonable con ~12,000 filas.


In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print('Dispositivo usado:', device)

if device == "cpu":
    print('⚠️ ADVERTENCIA: no se detectó GPU. Esto será muy lento con ~12,000 filas.')
    print('Activa GPU en: Entorno de ejecución → Cambiar tipo de entorno → GPU T4')

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Dispositivo usado: cuda


## 2. Función de traducción por trozos

NLLB-200 tiene un límite de tokens por petición (512). Los textos largos
se dividen en trozos de palabras, se traduce cada uno por separado, y se
vuelven a unir — así no se pierde contenido en las filas más largas.

In [17]:
def traducir_largo(texto, max_words=300):
    texto = str(texto)
    palabras = texto.split(' ')
    trozos = [' '.join(palabras[i:i+max_words]) for i in range(0, len(palabras), max_words)]

    traducidos = []
    for trozo in trozos:
        inputs = tokenizer(trozo, return_tensors="pt", truncation=True, max_length=512).to(device)
        salida = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("spa_Latn"),
            max_length=512
        )
        traducidos.append(tokenizer.decode(salida[0], skip_special_tokens=True))
    return ' '.join(traducidos)

In [18]:
muestra_test = df.head(5).copy()
muestra_test['text_es'] = muestra_test['text'].apply(traducir_largo)

for i, row in muestra_test.iterrows():
    print(f"--- Fila {i} (label={row['label']}) ---")
    print("EN:", row['text'][:200])
    print("ES:", row['text_es'][:200])
    print()

--- Fila 0 (label=1) ---
EN: Get Free UNI Tokens From The V4 Protocol Reward Program Presenting: V4 Protocol Reward Program (v4PRP) The DECC upgrade brings forth significantly reduced gas costs and the implementation of the V4 Pr
ES: Obtenga tokens gratuitos de UNI del Programa de Recompensa del Protocolo V4 Presentación: Programa de Recompensa del Protocolo V4 (v4PRP) La actualización de DECC trae un costo de gas significativamen

--- Fila 1 (label=0) ---
EN: GTV weekly project status update Please find below the weekly project status update for GTV and the attached file for the updated project timelines. Changes are underlined. GTV status meeting Dang, Na
ES: GTV actualización semanal del estado del proyecto Por favor, encuentre a continuación la actualización semanal del estado del proyecto para GTV y el archivo adjunto para las líneas de tiempo de proyec

--- Fila 2 (label=0) ---
EN: JM Huber I haven't answered him yet. I just plan to let him know that the contracts should be execu

## 3b. Comparación con Helsinki-NLP (OPUS-MT)

Se traduce el mismo set de 5 filas con un segundo modelo (Helsinki-NLP
opus-mt-en-es) para comparar calidad lado a lado con NLLB-200, antes de
decidir cuál usar para el dataset completo.

In [19]:
from transformers import MarianMTModel, MarianTokenizer

model_name_helsinki = "Helsinki-NLP/opus-mt-en-es"
tokenizer_helsinki = MarianTokenizer.from_pretrained(model_name_helsinki)
model_helsinki = MarianMTModel.from_pretrained(model_name_helsinki).to(device)

def traducir_helsinki(texto, max_words=300):
    texto = str(texto)
    palabras = texto.split(' ')
    trozos = [' '.join(palabras[i:i+max_words]) for i in range(0, len(palabras), max_words)]

    traducidos = []
    for trozo in trozos:
        inputs = tokenizer_helsinki(trozo, return_tensors="pt", truncation=True, max_length=512).to(device)
        salida = model_helsinki.generate(**inputs, max_length=512)
        traducidos.append(tokenizer_helsinki.decode(salida[0], skip_special_tokens=True))
    return ' '.join(traducidos)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [20]:
from tqdm.notebook import tqdm

textos_helsinki = []
for texto in tqdm(muestra_test['text'], desc="Traduciendo (Helsinki)"):
    textos_helsinki.append(traducir_helsinki(texto))

muestra_test['text_es_helsinki'] = textos_helsinki

for i, row in muestra_test.iterrows():
    print(f"--- Fila {i} (label={row['label']}) ---")
    print("EN        :", row['text'][:180])
    print("ES (NLLB) :", row['text_es'][:180])
    print("ES (Helsinki):", row['text_es_helsinki'][:180])
    print()

Traduciendo (Helsinki):   0%|          | 0/5 [00:00<?, ?it/s]

--- Fila 0 (label=1) ---
EN        : Get Free UNI Tokens From The V4 Protocol Reward Program Presenting: V4 Protocol Reward Program (v4PRP) The DECC upgrade brings forth significantly reduced gas costs and the impleme
ES (NLLB) : Obtenga tokens gratuitos de UNI del Programa de Recompensa del Protocolo V4 Presentación: Programa de Recompensa del Protocolo V4 (v4PRP) La actualización de DECC trae un costo de 
ES (Helsinki): Obtener gratis UNI Tokens from The V4 Protocol Reward Program Presentation: V4 Protocol Reward Program (v4PRP) La actualización de DECC trae consigo una reducción significativa de 

--- Fila 1 (label=0) ---
EN        : GTV weekly project status update Please find below the weekly project status update for GTV and the attached file for the updated project timelines. Changes are underlined. GTV sta
ES (NLLB) : GTV actualización semanal del estado del proyecto Por favor, encuentre a continuación la actualización semanal del estado del proyecto para GTV y el archivo adjun

## 4. Traducción del dataset completo con NLLB-200

Se aplica la función `traducir_largo` (NLLB-200) sobre las ~12,000 filas
completas, con barra de progreso para monitorear el avance. Esto puede
tardar bastante tiempo incluso con GPU — se recomienda guardar progreso
parcial cada cierto número de filas, para no perder el trabajo si la
sesión de Colab se desconecta.

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd

GUARDADO_CADA = 500  # guarda progreso parcial cada 500 filas traducidas
RUTA_PARCIAL = '/content/drive/MyDrive/dataset_traduccion_parcial.csv'

textos_traducidos = []
for i, texto in enumerate(tqdm(df['text'], desc="Traduciendo dataset completo")):
    textos_traducidos.append(traducir_largo(texto))

    if (i + 1) % GUARDADO_CADA == 0:
        df_parcial = df.iloc[:i+1].copy()
        df_parcial['text_es'] = textos_traducidos
        df_parcial.to_csv(RUTA_PARCIAL, index=False, encoding='utf-8-sig')

df['text_es'] = textos_traducidos
print('Traducción completa.')

Traduciendo dataset completo:   0%|          | 0/9534 [00:00<?, ?it/s]

In [ ]:
ruta_salida_final = '/content/drive/MyDrive/dataset_traducido_es.csv'
df.to_csv(ruta_salida_final, index=False, encoding='utf-8-sig')
print('Guardado en:', ruta_salida_final, '| filas:', len(df))